# 02 Asteria Multi-Simulation

Runs a batch of Asteria simulations with different configurations (noise levels, environments, rail parameters).
Each run is exported to its own subdirectory under `simulated_data/rocketpy/`.

In [ ]:
from dataclasses import dataclass, field, replace

from matplotlib import pyplot as plt

from core import SimulationResult
from pipeline.dropout import DropoutInjector
from pipeline.noise import NoiseInjector
from pipeline.noise.core.noise_profile import NoiseProfile
from pipeline.synthetic import SyntheticMagnetometer, SyntheticSensorGenerator
from rocketpy_simulation import RocketSimulation, EnvironmentBase
from rocketpy_simulation.asteria import AsteriaMotor, AsteriaRocket
from rocketpy_simulation.environments import EuroCEnvironment

In [ ]:
BASE_DATA_PATH: str = "../../simulated_data/rocketpy/"
plt.style.use('default')

In [ ]:
from rocketpy_simulation.environments import EuroCEnvironmentNoWind, EuroCEnvironmentWindy, EuroCEnvironmentSlightWind, \
    EuroCEnvironmentGust, EuroCEnvironmentHotDay, EuroCEnvironmentColdDay


@dataclass
class SimulationRun:
    name: str
    accel_noise_multiplier: float = 1.0
    gyro_noise_multiplier: float = 1.0
    baro_noise_multiplier: float = 1.0
    magnetometer_noise_multiplier: float = 1.0
    gnss_noise_multiplier: float = 1.0
    environment: EnvironmentBase = field(default_factory=EuroCEnvironment)
    rail_inclination: float = 85.0
    rail_heading: float = 144.0
    include_parachutes: bool = True
    gnss_dropout_probability: float = 0.0
    gnss_dropout_time: list[tuple[float, float]] | None = None
    gnss_dropout_accel_g: float | None = None


## 1. Run Configurations

In [ ]:
ASTERIA_RUNS: list[SimulationRun] = [
    # Low-noise baseline
    SimulationRun(
        name="low_noise_baseline",
        accel_noise_multiplier=0.5,
        gyro_noise_multiplier=0.5,
        gnss_noise_multiplier=0.5,
        environment=EuroCEnvironment(),
        rail_inclination=84.0,
        rail_heading=130,
        gnss_dropout_probability=0.0,
        gnss_dropout_time=None,
        gnss_dropout_accel_g=None,
    ),

    # ballistic descent (parachute failure)
    SimulationRun(
        name="ballistic_descent",
        accel_noise_multiplier=1.0,
        gyro_noise_multiplier=1.0,
        gnss_noise_multiplier=1.0,
        environment=EuroCEnvironment(),
        rail_inclination=85.0,
        rail_heading=135.0,
        gnss_dropout_probability=0.0,
        gnss_dropout_time=None,
        gnss_dropout_accel_g=4,
        include_parachutes=False,
    ),

    # High IMU/Baro noise
    SimulationRun(
        name="high_sensor_noise",
        accel_noise_multiplier=1.5,
        gyro_noise_multiplier=1.5,
        gnss_noise_multiplier=1.0,
        environment=EuroCEnvironmentNoWind(),
        rail_inclination=84.5,
        rail_heading=140.0,
        gnss_dropout_probability=0.0,
        gnss_dropout_time=None,
        gnss_dropout_accel_g=None,
    ),

    # Slight Winds + Faulty GNSS
    SimulationRun(
        name="slight_winds_and_bad_gnss",
        accel_noise_multiplier=1.0,
        gyro_noise_multiplier=1.0,
        gnss_noise_multiplier=1.5,
        environment=EuroCEnvironmentSlightWind(),
        rail_inclination=85.0,
        rail_heading=145.0,
        gnss_dropout_probability=0.5,
        gnss_dropout_time=[(2.0, 40.0)],
        gnss_dropout_accel_g=None,
    ),

    # Severe wind
    SimulationRun(
        name="high_winds",
        accel_noise_multiplier=1.0,
        gyro_noise_multiplier=1.0,
        gnss_noise_multiplier=1.0,
        environment=EuroCEnvironmentWindy(),
        rail_inclination=85.5,
        rail_heading=150.0,
        gnss_dropout_probability=0.0,
        gnss_dropout_time=None,
        gnss_dropout_accel_g=None,
    ),

    # Gusty wind + GNSS high-g dropout
    SimulationRun(
        name="gust_and_gnss_launch_dropout",
        accel_noise_multiplier=1.0,
        gyro_noise_multiplier=1.0,
        gnss_noise_multiplier=1.0,
        environment=EuroCEnvironmentGust(),
        rail_inclination=86.0,
        rail_heading=155.0,
        gnss_dropout_probability=0,
        gnss_dropout_time=None,
        gnss_dropout_accel_g=4.0,
    ),

    # Hot day + GNSS high-g dropout
    SimulationRun(
        name="hot_day_bad_gnss",
        accel_noise_multiplier=1.0,
        gyro_noise_multiplier=1.0,
        gnss_noise_multiplier=1.5,
        environment=EuroCEnvironmentHotDay(),
        rail_inclination=86.5,
        rail_heading=160.0,
        gnss_dropout_probability=0.0,
        gnss_dropout_time=None,
        gnss_dropout_accel_g=4.0,
    ),

    # Cold day + High noise + GNSS launch drop
    SimulationRun(
        name="cold_day_high_noise",
        accel_noise_multiplier=1.2,
        gyro_noise_multiplier=1.2,
        gnss_noise_multiplier=1.0,
        environment=EuroCEnvironmentColdDay(),
        rail_inclination=87.0,
        rail_heading=165.0,
        gnss_dropout_probability=0.0,
        gnss_dropout_time=None,
        gnss_dropout_accel_g=4.0,
    ),

    # Evaluation Flight
    SimulationRun(
        name="asteria_evaluation",
        accel_noise_multiplier=1.0,
        gyro_noise_multiplier=1.0,
        gnss_noise_multiplier=1.0,
        environment=EuroCEnvironment(),
        rail_inclination=88.0,
        rail_heading=170.0,
        gnss_dropout_probability=0.0,
        gnss_dropout_time=[(2.0, 40.0)],
        gnss_dropout_accel_g=None,
    ),
]

In [ ]:
for run in ASTERIA_RUNS:
    print(run)

## 2. Setup

In [ ]:
accel_profile = NoiseProfile.from_yaml("../../pipeline/noise/profiles/LSM6DSO32_accelerometer.yaml")
gyro_profile = NoiseProfile.from_yaml("../../pipeline/noise/profiles/LSM6DSO32_gyroscope.yaml")
baro_profile = NoiseProfile.from_yaml("../../pipeline/noise/profiles/MS5607_barometer.yaml")
mag_profile = NoiseProfile.from_yaml("../../pipeline/noise/profiles/LSM303AGR_eCompass_magnetometer.yaml")


def scale_noise(profile: NoiseProfile, multiplier: float) -> NoiseProfile:
    return replace(
        profile,
        noise_density=profile.noise_density * multiplier,
        bias_fixed=profile.bias_fixed * multiplier,
        random_walk=profile.random_walk * multiplier,
    )

## 3. Execute All Runs
For each run: simulate, add synthetic sensors, inject noise, export.

In [ ]:
import numpy as np
import scipy.constants
from pipeline.dropout.core.dropout_strategy import RandomDropout, WindowDropout, ExternalSignalThresholdDropout
from pipeline.dropout import DropoutMode

results: dict[str, SimulationResult] = {}

for run in ASTERIA_RUNS:
    print(f"\n{'=' * 60}")
    print(f"Run: {run.name}")
    print(f"{'=' * 60}")

    run_path = BASE_DATA_PATH + run.name

    # --- 1. Object Generation & Run ---
    motor = AsteriaMotor().build()
    simulation = RocketSimulation(
        environment_builder=run.environment,
        rocket_builder=AsteriaRocket(
            motor=motor,
            include_parachutes=run.include_parachutes,
            horizontal_position_accuracy=1.5 * run.gnss_noise_multiplier,
            vertical_position_accuracy=2.25 * run.gnss_noise_multiplier,
        ),
        inclination=run.rail_inclination,
        heading=run.rail_heading,
    )
    result = simulation.run(run.name)

    # --- 2. Synthetic Sensor Addition ---
    generator = SyntheticSensorGenerator(sensors=[
        (
            SyntheticMagnetometer(name="magnetometer", launch_site_elevation_m=EuroCEnvironment.elevation),
            NoiseInjector(scale_noise(mag_profile, run.magnetometer_noise_multiplier)),
            None,
        ),
    ])
    generator.apply(result)

    # --- 3. Noise Injection ---
    result.add_sensor_noisy("accelerometer",
                            NoiseInjector(scale_noise(accel_profile, run.accel_noise_multiplier)).apply(
                                result.sensors_clean["accelerometer_clean"]))
    result.add_sensor_noisy("gyroscope",
                            NoiseInjector(scale_noise(gyro_profile, run.gyro_noise_multiplier)).apply(
                                result.sensors_clean["gyroscope_clean"]))
    result.add_sensor_noisy("barometer",
                            NoiseInjector(scale_noise(baro_profile, run.baro_noise_multiplier)).apply(
                                result.sensors_clean["barometer_clean"]))

    # --- 4. Dropout Simulation ---
    strategies = []
    if run.gnss_dropout_probability > 0:
        strategies.append(RandomDropout(probability=run.gnss_dropout_probability))
    if run.gnss_dropout_time:
        strategies.append(WindowDropout(windows=run.gnss_dropout_time))
    if run.gnss_dropout_accel_g is not None:
        accel_mag = np.linalg.norm(result.ground_truth.acceleration, axis=1)
        strategies.append(ExternalSignalThresholdDropout(
            reference_time=result.ground_truth.time,
            reference_values=accel_mag,
            threshold=run.gnss_dropout_accel_g * scipy.constants.g,
        ))
    if strategies:
        dropout_injector = DropoutInjector(strategies=strategies, mode=DropoutMode.NAN)
        result.sensors_noisy["gnss"] = dropout_injector.apply(result.sensors_noisy["gnss"])

    # --- 5. Export ---
    simulation.flight.plots.trajectory_3d()
    simulation.flight.info()
    result.export_csv_data(run_path)
    result.save(f"{run_path}/{run.name}.pkl")

    results[run.name] = result

print("\nAll runs complete.")

## 4. Inspect

In [ ]:
import plotly.graph_objects as go
import plotly.colors

labels = [SimulationRun.name for SimulationRun in ASTERIA_RUNS]
fig = go.Figure()

colors = plotly.colors.qualitative.Plotly

for i, label in enumerate(labels):
    pos = results[label].ground_truth.position  # (N, 3) ENU
    trace_color = colors[i % len(colors)]

    fig.add_trace(go.Scatter3d(
        x=pos[:, 0],
        y=pos[:, 1],
        z=pos[:, 2],
        mode="lines",
        name=label,
        line=dict(width=4, color=trace_color),
        legendgroup=label
    ))

    fig.add_trace(go.Scatter3d(
        x=[pos[0, 0]],
        y=[pos[0, 1]],
        z=[pos[0, 2]],
        mode="markers",
        marker=dict(size=4, color=trace_color, symbol="circle"),
        showlegend=False,
        legendgroup=label,
        hoverinfo="skip"
    ))

# Set labels, title and equal axis scaling
fig.update_layout(
    title="Flight trajectories",
    scene=dict(
        xaxis_title="East [m]",
        yaxis_title="North [m]",
        zaxis_title="Up [m]",
        aspectmode="data"
    ),
    margin=dict(l=0, r=0, b=0, t=50)
)

fig.show()